In [ ]:
import numpy as np
import scipy.sparse as sp

# XXX These are private helper functions from scikit-learn. We should copy
# the code over instead of importing them.
from sklearn.utils import check_random_state
from sklearn.utils.extmath import safe_sparse_dot

from .ctrans import count_trans
from .safeadd import safe_add
from .transmatrix import make_trans_matrix

In [ ]:
cimport cython
cimport numpy as np
import numpy as np

np.import_array()

# TODO handle second-order transitions (trigrams)
@cython.boundscheck(False)
@cython.wraparound(False)
def count_trans(np.ndarray[ndim=1, dtype=np.npy_intp] y, n_classes):
    """Count transitions in a target vector.

    Parameters
    ----------
    y : array of integers, shape = n_samples
    n_classes : int
        Number of distinct labels.
    """
    cdef np.ndarray[ndim=2, dtype=np.npy_intp, mode='c'] trans
    cdef np.npy_intp i

    trans = np.zeros((n_classes, n_classes), dtype=np.intp)

    for i in range(y.shape[0] - 1):
        trans[y[i], y[i + 1]] += 1
    return trans

In [ ]:
def _assert_all_finite(X):
    X = np.asanyarray(X)
    # First try an O(n) time, O(1) space solution for the common case that
    # there everything is finite; fall back to O(n) space np.isfinite to
    # prevent false positives from overflow in sum method.
    if (X.dtype.char in np.typecodes['AllFloat'] and not np.isfinite(X.sum())
            and not np.isfinite(X).all()):
        raise ValueError("Input contains NaN, infinity"
                         " or a value too large for %r." % X.dtype)
    
def array2d(X, dtype=None, order=None, copy=False):
    """Returns at least 2-d array with data from X"""
    if sp.issparse(X):
        raise TypeError('A sparse matrix was passed, but dense data '
                        'is required. Use X.toarray() to convert to dense.')
    X_2d = np.asarray(np.atleast_2d(X), dtype=dtype, order=order)
    _assert_all_finite(X_2d)
    if X is X_2d and copy:
        X_2d = safe_copy(X_2d)
    return X_2d


def _atleast2d_or_sparse(X, dtype, order, copy, sparse_class, convmethod,
                         check_same_type):
    if sp.issparse(X):
        if check_same_type(X) and X.dtype == dtype:
            X = getattr(X, convmethod)(copy=copy)
        elif dtype is None or X.dtype == dtype:
            X = getattr(X, convmethod)()
        else:
            X = sparse_class(X, dtype=dtype)
        _assert_all_finite(X.data)
        X.data = np.array(X.data, copy=False, order=order)
    else:
        X = array2d(X, dtype=dtype, order=order, copy=copy)
    return X


def atleast2d_or_csr(X, dtype=None, order=None, copy=False):
    """Like numpy.atleast_2d, but converts sparse matrices to CSR format

    Also, converts np.matrix to np.ndarray.
    """
    return _atleast2d_or_sparse(X, dtype, order, copy, sp.csr_matrix,
                                "tocsr", sp.isspmatrix_csr)

In [2]:
from scipy.misc import logsumexp
import numpy as np

ImportError: cannot import name 'logsumexp' from 'scipy.misc' (C:\Users\sures\AppData\Roaming\Python\Python311\site-packages\scipy\misc\__init__.py)

In [3]:
class BaseSequenceClassifier(BaseEstimator, ClassifierMixin):
    def _get_decoder(self):
        try:
            return DECODERS[self.decode]
        except KeyError:
            raise ValueError("Unknown decoder {0!r}".format(self.decode))

    def predict(self, X, lengths=None):
        """Predict labels/tags for samples X.

        Parameters
        ----------
        X : {array-like, sparse matrix}, shape (n_samples, n_features)
            Feature matrix.

        lengths : array-like of integer, shape (n_sequences,), optional
            Lengths of sequences in X. If not given, X is assumed to be a
            single sequence of length n_samples.

        Returns
        -------
        y : array, shape (n_samples,)
            Labels per sample in X.
        """
        X = atleast2d_or_csr(X)
        scores = safe_sparse_dot(X, self.coef_.T)
        if hasattr(self, "coef_trans_"):
            n_classes = len(self.classes_)
            coef_t = self.coef_trans_.T.reshape(-1, self.coef_trans_.shape[-1])
            trans_scores = safe_sparse_dot(X, coef_t.T)
            trans_scores = trans_scores.reshape(-1, n_classes, n_classes)
        else:
            trans_scores = None

        decode = self._get_decoder()

        if lengths is None:
            y = decode(scores, trans_scores, self.intercept_trans_,
                       self.intercept_init_, self.intercept_final_)
        else:
            start, end = validate_lengths(X.shape[0], lengths)

            y = [decode(scores[start[i]:end[i]], trans_scores,
                        self.intercept_trans_, self.intercept_init_,
                        self.intercept_final_)
                 for i in six.moves.xrange(len(lengths))]
            y = np.hstack(y)

        return self.classes_[y]


    def score(self, X, y, lengths=None):
        """Returns the mean accuracy on the given test data and labels.

        Parameters
        ----------
        X : array-like, shape = (n_samples, n_features)
            Test samples.

        y : array-like, shape = (n_samples,)
            True labels for X.

        lengths : array-like of integer, shape (n_sequences,), optional
            Lengths of sequences in X. If not given, X is assumed to be a
            single sequence of length n_samples.

        Returns
        -------
        score : float
            Mean accuracy of self.predict(X, lengths) wrt. y.
        """
        return accuracy_score(y, self.predict(X, lengths))

NameError: name 'BaseEstimator' is not defined

In [ ]:
class MultinomialHMM(BaseSequenceClassifier):
    """First-order hidden Markov model with multinomial event model.

    Parameters
    ----------
    decode : string, optional
        Decoding algorithm, either "bestfirst" or "viterbi" (default).
        Best-first decoding is also called posterior decoding in the HMM
        literature.

    alpha : float
        Lidstone (additive) smoothing parameter.
    """

    def __init__(self, decode="viterbi", alpha=.01):
        self.alpha = alpha
        self.decode = decode

    def fit(self, X, y, lengths):
        """Fit HMM model to data.

        Parameters
        ----------
        X : {array-like, sparse matrix}, shape (n_samples, n_features)
            Feature matrix of individual samples.

        y : array-like, shape (n_samples,)
            Target labels.

        lengths : array-like of integers, shape (n_sequences,)
            Lengths of the individual sequences in X, y. The sum of these
            should be n_samples.

        Notes
        -----
        Make sure the training set (X) is one-hot encoded; if more than one
        feature in X is on, the emission probabilities will be multiplied.

        Returns
        -------
        self : MultinomialHMM
        """

        alpha = self.alpha
        if alpha <= 0:
            raise ValueError("alpha should be >0, got {0!r}".format(alpha))

        X = atleast2d_or_csr(X)
        classes, y = np.unique(y, return_inverse=True)
        lengths = np.asarray(lengths)
        Y = y.reshape(-1, 1) == np.arange(len(classes))

        end = np.cumsum(lengths)
        start = end - lengths

        init_prob = np.log(Y[start].sum(axis=0) + alpha)
        init_prob -= logsumexp(init_prob)
        final_prob = np.log(Y[start].sum(axis=0) + alpha)
        final_prob -= logsumexp(final_prob)

        feature_prob = np.log(safe_sparse_dot(Y.T, X) + alpha)
        feature_prob -= logsumexp(feature_prob, axis=0)

        trans_prob = np.log(count_trans(y, len(classes)) + alpha)
        trans_prob -= logsumexp(trans_prob, axis=0)

        self.coef_ = feature_prob
        self.intercept_init_ = init_prob
        self.intercept_final_ = final_prob
        self.intercept_trans_ = trans_prob

        self.classes_ = classes

        return self